# Milestone 2 — Hyperparameter Tuning (GPU-accelerated)
Tune gradient boosting with `RandomizedSearchCV` and **Optuna**, and test a `log1p` target transform. Tuning here runs on the **GPU** via **CatBoost** (`task_type="GPU"`). Because there is a single GPU, the search/CV run sequentially (`n_jobs=1`) while each fit is GPU-accelerated.

*(LightGBM — the final production model — is tuned on CPU; its pip wheel needs OpenCL for GPU, which is unavailable under WSL. XGBoost and CatBoost are the GPU-capable libraries here.)*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from scipy.stats import randint, loguniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, KFold
from sklearn.compose import TransformedTargetRegressor
from catboost import CatBoostRegressor
from insurance import config, data, evaluate
from insurance.pipeline import build_pipeline

df = data.load_clean()
X, y = data.split_X_y(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE)

def cat_gpu(**kw):
    # GPU training; search/CV use n_jobs=1 (single GPU). Predict runs on CPU cleanly.
    return CatBoostRegressor(random_state=config.RANDOM_STATE, task_type="GPU", devices="0",
                             verbose=0, allow_writing_files=False, **kw)

print("Training device: GPU (CatBoost task_type=GPU)")

Training device: GPU (CatBoost task_type=GPU)


## 1. RandomizedSearchCV on GPU (CatBoost)

In [2]:
space = {
    "model__iterations": randint(200, 500),
    "model__depth": randint(4, 10),
    "model__learning_rate": loguniform(1e-2, 3e-1),
    "model__l2_leaf_reg": loguniform(1.0, 10.0),
}
search = RandomizedSearchCV(
    build_pipeline(cat_gpu()), space, n_iter=10, cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=config.RANDOM_STATE, n_jobs=1)
search.fit(X_train, y_train)
m = evaluate.evaluate_train_test(search.best_estimator_, X_train, y_train, X_test, y_test)
print(f"Best CV RMSE: {-search.best_score_:.1f} | test RMSE: {m['test_RMSE']:.1f} | test R2: {m['test_R2']:.4f}")
print("Best params:", {k.replace('model__', ''): (round(v, 4) if isinstance(v, float) else v) for k, v in search.best_params_.items()})

Best CV RMSE: 2983.3 | test RMSE: 2943.5 | test R2: 0.9575
Best params: {'depth': 6, 'iterations': 254, 'l2_leaf_reg': np.float64(9.6212), 'learning_rate': np.float64(0.0489)}


## 2. Optuna study on GPU
Bayesian search optimising 3-fold CV RMSE, each trial trained on the GPU.

In [3]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
kf = KFold(n_splits=3, shuffle=True, random_state=config.RANDOM_STATE)

def objective(trial):
    params = dict(
        iterations=trial.suggest_int("iterations", 200, 500),
        depth=trial.suggest_int("depth", 4, 10),
        learning_rate=trial.suggest_float("learning_rate", 1e-2, 3e-1, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
    )
    pipe = build_pipeline(cat_gpu(**params))
    scores = cross_val_score(pipe, X_train, y_train, scoring="neg_root_mean_squared_error", cv=kf, n_jobs=1)
    return -scores.mean()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10, show_progress_bar=False)
print("Best CV RMSE:", round(study.best_value, 1))
print("Best params:", study.best_params)

Best CV RMSE: 2981.6
Best params: {'iterations': 298, 'depth': 7, 'learning_rate': 0.03687913866924263, 'l2_leaf_reg': 1.0433254011574251}


## 3. Target transform experiment
`insurance_cost` is mildly right-skewed — does a `log1p` target transform help? (GPU-trained CatBoost.)

In [4]:
base = build_pipeline(cat_gpu())
logged = TransformedTargetRegressor(regressor=base, func=np.log1p, inverse_func=np.expm1)
for label, model in [("identity target", base), ("log1p target", logged)]:
    model.fit(X_train, y_train)
    mm = evaluate.regression_metrics(y_test, model.predict(X_test))
    print(f"{label:16s} test RMSE={mm['RMSE']:.1f}  R2={mm['R2']:.4f}  MAPE={mm['MAPE']:.2f}")

identity target  test RMSE=2929.3  R2=0.9579  MAPE=11.30


log1p target     test RMSE=2945.4  R2=0.9574  MAPE=11.01


**Observation:** GPU-accelerated CatBoost tuning reaches test R² ≈ 0.958 — on par with the untuned gradient-boosting models — confirming the dataset is well-modelled by boosting and that careful tuning yields only marginal gains. The log-target transform gives no meaningful improvement (the skew is mild). LightGBM (CPU) remains the production choice on a razor-thin margin; CatBoost-GPU is an equally valid, GPU-trained alternative. The production trainer (`insurance.train`) trains XGBoost and CatBoost on the GPU automatically when one is present.